# 🌊 AquaGuard AI — Day 09: Temporal Split Definition & Quarantined Lockbox
- **Date:** 28 Aug 2026
- **Phase:** Phase 1: Ingestion & Governance
- **Objective:** Partition telemetry stream into 70% Train, 15% Validation, and 15% Test Lockbox. Quarantine Test set.
- **Inputs Consumed:** `data/raw/underwater_sensor_stream.csv`, `artifacts/day08/rules.yaml`
- **Outputs Produced:** `artifacts/day09/split_metadata.json`, `artifacts/day09/split.json`
- **Deliverable Satisfied:** Chronological split boundaries with zero temporal leakage.

In [16]:
import os, sys
from pathlib import Path
# Universal project root detection (works from any clone directory or depth)
def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import numpy as np
import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_raw_data, load_artifact, save_artifact
from src.preprocessing import temporal_walk_forward_split, assert_no_leakage

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day09")

# Prerequisite check
rules = load_artifact("artifacts/day08/rules.yaml", required_by_day="Day 08")
print("Prerequisite verified: Day 08 rules loaded.")

Prerequisite verified: Day 08 rules loaded.


## 1. Chronological Walk-Forward Partitioning
We split the stream chronologically without shuffling to reflect real-world operational forecasting.

In [17]:
df_stream = load_raw_data(config["paths"]["raw_temporal_data"])
ts_col = config["track_a"]["timestamp_column"]

train_df, val_df, test_df = temporal_walk_forward_split(
    df_stream,
    timestamp_col=ts_col,
    train_ratio=config["track_a"]["split_boundaries"]["train_ratio"],
    val_ratio=config["track_a"]["split_boundaries"]["val_ratio"],
    test_ratio=config["track_a"]["split_boundaries"]["test_ratio"]
)

# Assert leakage boundaries
leakage_audit = assert_no_leakage(train_df, val_df, test_df, timestamp_col=ts_col)
print("Leakage Assertion Status:", leakage_audit)

2026-10-19 17:39:03 | INFO    | IO - Loaded raw data from data\raw\underwater_sensor_stream.csv: 720 rows, 7 columns
2026-10-19 17:39:03 | INFO    | Preprocessing - Chronological split: Train=503, Val=109, Test Lockbox=108


Leakage Assertion Status: {'no_train_val_overlap': True, 'no_val_test_overlap': True, 'disjoint_indices': True}


## 2. Export Partition Metadata & Index Maps
Save split boundaries and record quarantine certificates.

In [18]:
split_meta = {
    "train": {
        "records": len(train_df),
        "start_time": str(train_df[ts_col].min()),
        "end_time": str(train_df[ts_col].max())
    },
    "val": {
        "records": len(val_df),
        "start_time": str(val_df[ts_col].min()),
        "end_time": str(val_df[ts_col].max())
    },
    "test_lockbox": {
        "records": len(test_df),
        "start_time": str(test_df[ts_col].min()),
        "end_time": str(test_df[ts_col].max()),
        "status": "QUARANTINED - DO NOT ACCESS UNTIL DAY 23"
    }
}

split_indices = {
    "train_indices": train_df.index.tolist(),
    "val_indices": val_df.index.tolist(),
    "test_indices": test_df.index.tolist()
}

save_artifact(split_meta, "artifacts/day09/split_metadata.json")
save_artifact(split_indices, "artifacts/day09/split.json")
print("Split metadata saved to artifacts/day09/split_metadata.json")
print("Partition summary:", split_meta)

2026-10-19 17:39:03 | INFO    | IO - Saved artifact to artifacts\day09\split_metadata.json
2026-10-19 17:39:03 | INFO    | IO - Saved artifact to artifacts\day09\split.json


Split metadata saved to artifacts/day09/split_metadata.json
Partition summary: {'train': {'records': 503, 'start_time': '2026-08-20 00:00:00', 'end_time': '2026-09-09 22:00:00'}, 'val': {'records': 109, 'start_time': '2026-09-09 23:00:00', 'end_time': '2026-09-14 11:00:00'}, 'test_lockbox': {'records': 108, 'start_time': '2026-09-14 12:00:00', 'end_time': '2026-09-18 23:00:00', 'status': 'QUARANTINED - DO NOT ACCESS UNTIL DAY 23'}}


## 📌 Day 09 Execution Findings & Sign-Off
- **Key Finding:** Strict walk-forward temporal order preserved. Test partition is officially sealed.
- **Artifacts Saved:** `artifacts/day09/split_metadata.json`, `artifacts/day09/split.json`.
- **Handoff:** Ready for Day 10 Environmental EDA (conducted on Train partition only).